# MSC — Supplementary Analysis: Time-Varying Material Identity

Set `QUICK = True` for a smoke test, `False` for production.

**Run order:** top to bottom. The two hard gates (Cell 6, Cell 6b) must print PASSED before the sweeps.

## Cell 1 — Mount Drive, set paths, persist working files

In [ ]:
# 1. Runtime-local setup and input upload
# Drive is not required. All outputs remain in /content.
import pathlib, shutil, sys, hashlib, json, datetime, zipfile
from google.colab import files

ROOT = pathlib.Path('/content/msc_material_dynamics/output')
PROJECT = ROOT / 'model'
DATA = ROOT / 'data'
FIG = ROOT / 'figures'
CHECKPOINTS = ROOT / 'checkpoints'
MANIFESTS = ROOT / 'manifests'
LOGS = ROOT / 'logs'
R_INPUT = ROOT / '10_r_input'
for path in (ROOT, PROJECT, DATA, FIG, CHECKPOINTS, MANIFESTS, LOGS, R_INPUT):
    path.mkdir(parents=True, exist_ok=True)

def sha256_file(path, chunk_size=1024*1024):
    digest=hashlib.sha256()
    with open(path,'rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''): digest.update(chunk)
    return digest.hexdigest()

def place(source,destination):
    source,destination=pathlib.Path(source),pathlib.Path(destination)
    if source.resolve()!=destination.resolve(): shutil.copy2(source,destination)

MODEL_UPLOAD=pathlib.Path('/content/msc_model_feedback.py')
CONFIG_UPLOAD=pathlib.Path('/content/config_msc_feedback.yaml')
if not (MODEL_UPLOAD.exists() and CONFIG_UPLOAD.exists()):
    print('Upload msc_model_feedback.py and config_msc_feedback.yaml now.')
    uploaded=files.upload()
    for name in uploaded:
        path=pathlib.Path('/content')/name
        if name.endswith('.py'): place(path,MODEL_UPLOAD)
        elif name.endswith(('.yaml','.yml')): place(path,CONFIG_UPLOAD)
if not MODEL_UPLOAD.exists() or not CONFIG_UPLOAD.exists():
    raise FileNotFoundError('Both msc_model_feedback.py and config_msc_feedback.yaml are required.')
place(MODEL_UPLOAD,PROJECT/'msc_model_feedback.py')
place(CONFIG_UPLOAD,PROJECT/'config_msc_feedback.yaml')
if str(PROJECT) not in sys.path: sys.path.insert(0,str(PROJECT))
print('Output root:',ROOT)
print('Model hash:',sha256_file(PROJECT/'msc_model_feedback.py')[:16])
print('Config hash:',sha256_file(PROJECT/'config_msc_feedback.yaml')[:16])


## Cell 2 — Dependencies and global switches

In [ ]:
!pip -q install pyyaml statsmodels pandas numpy matplotlib scipy joblib

import numpy as np, pandas as pd, matplotlib.pyplot as plt, json, time, math
from joblib import Parallel, delayed
from scipy.stats import wilcoxon

QUICK   = False
N_SEEDS = 6 if QUICK else 100
N_JOBS  = -1
VARIANT = 'full_msc_feedback'
EQUIL_MAX = 0.10          # tau_m <= 10% of T_total counts as equilibrated
np.set_printoptions(precision=4, suppress=True)
print('QUICK =', QUICK, '| N_SEEDS =', N_SEEDS)

## Cell 3 — Write the extension module (exact OU, unchanged from v5)

`%%writefile` magic, so no string quoting is needed. Path is hard-coded because the magic cannot take a Python variable — edit the first line if your `PROJECT` differs.

In [ ]:
%%writefile /content/msc_material_dynamics/output/model/msc_material_dynamics.py
"""MSC material-dynamics extension.

Adds time-varying material identity to the MSC feedback model WITHOUT touching
msc_model_feedback.py. Implements:

  (1) Ornstein-Uhlenbeck drift of the granule material vector u_g(t)
  (2) Activity-dependent drift of the synaptic target vector theta_i(t)
  (3) Transcript copy number N rescaling material spread as sigma/sqrt(N)

Design invariants
-----------------
* All drift stochasticity is drawn from a SEPARATE generator (rng_drift).
  The main `rng` stream is consumed in exactly the same order as in
  msc_model_feedback.simulate_one, so with drift disabled the output is
  bit-identical to the unmodified model.
* Intrinsic productive capacity q_g is fixed at granule birth from u_true(0).
  Sequence identity is immutable; only the material state drifts.
* Copy number rescales strong_sigma / weak_sigma only. weak_shift is held
  fixed, so the effect is pure variance, not mean proximity.
* The OU step uses the EXACT discretization
      u <- u0 + (u - u0)*a + b*xi,   a = exp(-dt/tau_m),  b = sigma_inf*sqrt(1-a^2)
  which is unconditionally stable and reproduces the stationary spread at every
  tau_m. The Euler-Maruyama form used previously diverged for dt/tau_m >~ 1
  (rho <~ 0.1 at dt = 2 s) and inflated sigma_inf by ~40% at dt/tau_m = 1.
"""
from __future__ import annotations

from typing import Any, Dict, List, Optional, Sequence
import math
import numpy as np
import pandas as pd

from msc_model_feedback import (
    MODEL_VARIANTS,
    axes_from_variant,
    material_distance2,
    reservoir_add,
)

DRIFT_DEFAULTS: Dict[str, Any] = {
    "tau_m": float("inf"),   # OU relaxation time of u_g (s)
    "sigma_rel": 0.0,        # stationary sd of drift, in units of strong_sigma
    "k_c": 0.0,              # activity-dependent theta drift rate (1/s)
    "n_copies": 1,           # transcript copies per granule
    "c_wt_scale": 0.6,       # magnitude of the WT-like target vector
}


def make_material_pool_copies(p: Dict[str, Any], rng: np.random.Generator,
                              n_copies: int = 1):
    """make_material_pool with sigma -> sigma/sqrt(N). Draw order preserved."""
    D = int(p["D_material"])
    Ngran = int(p["Ngran"])
    sc = 1.0 / math.sqrt(max(1, int(n_copies)))
    n_strong = int(np.clip(round(Ngran * p["strong_fraction"]), 1, Ngran - 1))
    u_strong = rng.normal(0.0, p["strong_sigma"] * sc, size=(n_strong, D))
    signs = rng.choice([-1.0, 1.0], size=(Ngran - n_strong, D))
    u_weak = signs * rng.normal(p["weak_shift"], p["weak_sigma"] * sc,
                                size=(Ngran - n_strong, D))
    u = np.vstack([u_strong, u_weak])
    perm = rng.permutation(Ngran)
    u = u[perm]
    d2_to_active = np.mean(u[:, : min(3, D)] ** 2, axis=1)
    q_g = np.exp(-0.5 * d2_to_active / (float(p["cargo_ell"]) ** 2))
    strong = q_g >= np.quantile(q_g, 1.0 - float(p["strong_fraction"]))
    return u, q_g.astype(float), strong.astype(bool)


def simulate_one_drift(
    params: Dict[str, Any],
    seed: int,
    variant_name: str = "full_msc_feedback",
    active_synapses: Optional[Sequence[int]] = None,
    activation_amp: Optional[float] = None,
    initial_S: Optional[Dict[int, float]] = None,
    initial_M: Optional[Dict[int, float]] = None,
    record_obs: bool = False,
    record_trace: bool = False,
    drift: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    if variant_name not in MODEL_VARIANTS:
        raise ValueError(f"Unknown variant {variant_name}")
    variant = MODEL_VARIANTS[variant_name]
    p = dict(params)
    dr = dict(DRIFT_DEFAULTS)
    dr.update(drift or {})

    rng = np.random.default_rng(int(seed))
    rng_drift = np.random.default_rng(int(seed) + 987_654_321)   # SEPARATE stream

    Nsyn = int(p["Nsyn"]); Ngran = int(p["Ngran"]); L = float(p["L"])
    Dm = int(p["D_material"]); dt = float(p["dt"])
    n_steps = int(round(float(p["T_total"]) / dt))
    activation_step = int(round(float(p["activation_time"]) / dt))
    delay_steps = max(1, int(round(float(p["delta_translation"]) / dt)))
    late_steps = max(1, int(round(float(p["late_window"]) / dt)))
    max_obs = int(p.get("max_obs_per_run", 5000))
    activation_amp = float(p["activation_amp"] if activation_amp is None else activation_amp)

    xsyn = np.linspace(L / (Nsyn + 1), L - L / (Nsyn + 1), Nsyn)
    if active_synapses is None:
        active_synapses = [Nsyn // 2]
    active_synapses = [int(x) for x in active_synapses]
    active_mask = np.zeros(Nsyn, dtype=bool)
    active_mask[active_synapses] = True

    theta = rng.normal(0.0, 1.0, size=(Nsyn, Dm))
    for i in active_synapses:
        theta[i, :] = 0.0

    u_true, q_g, strong = make_material_pool_copies(p, rng, int(dr["n_copies"]))
    u0 = u_true.copy()
    if variant["mode"] == "shuffled":
        perm_shuf = rng.permutation(Ngran)
        u_used = u_true[perm_shuf]
    elif variant["mode"] == "random":
        perm_shuf = None
        u_used = rng.normal(0.0, 1.25, size=u_true.shape)
    else:
        perm_shuf = None
        u_used = u_true.copy()

    gpos = rng.uniform(0.0, L, size=Ngran)
    gdir = rng.choice([-1.0, 1.0], size=Ngran)

    S = np.zeros(Nsyn); M = np.full(Nsyn, float(p["M_bas"])); Ttag = np.zeros(Nsyn)
    if initial_S:
        for k, v in initial_S.items(): S[int(k)] = float(v)
    if initial_M:
        for k, v in initial_M.items(): M[int(k)] = float(v)

    pending = np.zeros((n_steps + delay_steps + 2, Nsyn))
    S_hist = np.zeros((n_steps, Nsyn), dtype=np.float32)
    M_hist = np.zeros((n_steps, Nsyn), dtype=np.float32)
    T_active_hist = np.zeros(n_steps, dtype=np.float32) if record_trace else None

    axes = axes_from_variant(variant, Dm)
    dock_rows: List[Dict[str, Any]] = []
    obs_rows: List[Dict[str, Any]] = []
    obs_seen = 0
    feedback_on = bool(variant["feedback"]); material_on = bool(variant["material"])
    Kn = float(p["K_M"]) ** int(p["n_hill"])

    # ---- DRIFT setup -------------------------------------------------
    tau_m = float(dr["tau_m"]); sigma_rel = float(dr["sigma_rel"])
    drift_u_on = np.isfinite(tau_m) and tau_m > 0.0 and sigma_rel > 0.0
    # Stationary spread held at sigma_inf = sigma_rel * strong_sigma, independent
    # of tau_m, so rho is a pure timescale axis. Exact OU discretization.
    sigma_inf = sigma_rel * float(p["strong_sigma"]) if drift_u_on else 0.0
    if drift_u_on:
        ou_a = math.exp(-dt / tau_m)
        ou_b = sigma_inf * math.sqrt(max(0.0, 1.0 - ou_a * ou_a))
        sigma_m = sigma_inf * math.sqrt(2.0 / tau_m)   # equivalent continuous-time sigma
    else:
        ou_a = 1.0; ou_b = 0.0; sigma_m = 0.0
    k_c = float(dr["k_c"]); drift_theta_on = k_c > 0.0
    c_wt = np.full(Dm, float(dr["c_wt_scale"]) / math.sqrt(Dm))
    # ------------------------------------------------------------------

    for step in range(n_steps):
        if step == activation_step:
            for i in active_synapses:
                Ttag[i] = min(1.0, Ttag[i] + activation_amp)

        S += pending[step]
        S += (-S / float(p["tau_S"])) * dt
        S = np.clip(S, 0.0, 1.0)

        if feedback_on:
            Sn = S ** int(p["n_hill"])
            H = Sn / (Kn + Sn + 1e-12)
            M += (-float(p["k_M"]) * (M - float(p["M_bas"]))
                  + float(p["alphaM"]) * H * (float(p["M_max"]) - M)) * dt
            M = np.clip(M, 0.0, 1.0)

        Ttag += (-Ttag / float(p["tau_T"])) * dt
        Ttag = np.clip(Ttag, 0.0, 1.0)

        gpos += gdir * float(p["v"]) * dt + np.sqrt(2.0 * float(p["D"]) * dt) * rng.standard_normal(Ngran)
        flips = rng.random(Ngran) < float(p["p_switch"])
        gdir[flips] *= -1.0
        below = gpos < 0.0; above = gpos > L
        gpos[below] = -gpos[below]; gpos[above] = 2.0 * L - gpos[above]
        gdir[below | above] *= -1.0

        # ---- DRIFT (1): granule material vector ----------------------
        if drift_u_on:
            u_true = u0 + (u_true - u0) * ou_a + ou_b * \
                     rng_drift.standard_normal((Ngran, Dm))
            if variant["mode"] == "shuffled":
                u_used = u_true[perm_shuf]
            elif variant["mode"] != "random":
                u_used = u_true
        # ---- DRIFT (2): synaptic target vector -----------------------
        if drift_theta_on:
            theta += k_c * S[:, None] * (c_wt[None, :] - theta) * dt
        # --------------------------------------------------------------

        for g in range(Ngran):
            near = np.flatnonzero(np.abs(xsyn - gpos[g]) <= float(p["epsilon"]))
            if near.size == 0:
                continue
            for i in near:
                d2 = material_distance2(u_used[g], theta[i], axes)
                if material_on:
                    phi_h = math.exp(-0.5 * d2 / (float(p["ell_h"]) ** 2))
                    phi_q = math.exp(-0.5 * d2 / (float(p["ell_q"]) ** 2))
                else:
                    phi_h = 1.0; phi_q = 1.0
                exponent = float(p["beta_T"]) * Ttag[i]
                if feedback_on:
                    exponent += float(p["beta_M"]) * M[i]
                hazard = float(p["h0"]) * math.exp(min(20.0, exponent)) * phi_h
                pdock = 1.0 - math.exp(-hazard * dt)
                dock = rng.random() < pdock

                if record_obs:
                    obs_seen = reservoir_add(
                        obs_rows,
                        {"seed": int(seed), "variant": variant_name,
                         "T": float(Ttag[i]), "M": float(M[i]), "d2": float(d2),
                         "d2_true": float(material_distance2(u_true[g], theta[i], axes)),
                         "d2_birth": float(material_distance2(u0[g], theta[i], axes)),
                         "active_synapse": int(active_mask[i]), "dock": int(dock)},
                        obs_seen, max_obs, rng)

                if dock:
                    qig = float(q_g[g] * phi_q)
                    productive = qig >= float(p["q_min"])
                    if productive:
                        deliver = min(step + delay_steps, pending.shape[0] - 1)
                        pending[deliver, i] += float(p["alphaS"]) * qig
                    dock_rows.append(
                        {"seed": int(seed), "variant": variant_name, "t": step * dt,
                         "synapse": int(i), "granule": int(g),
                         "active_synapse": int(active_mask[i]), "d2": float(d2),
                         "q_g": float(q_g[g]), "q_ig": float(qig),
                         "strong": int(strong[g]), "productive": int(productive),
                         "T": float(Ttag[i]), "M": float(M[i])})

        S_hist[step] = S
        M_hist[step] = M
        if record_trace:
            T_active_hist[step] = float(Ttag[active_mask].mean())

    late_slice = slice(max(0, n_steps - late_steps), n_steps)
    active_S_late = float(S_hist[late_slice][:, active_mask].mean())
    control_late = S_hist[late_slice][:, ~active_mask]
    control_max_S_late = float(control_late.mean(axis=0).max()) if control_late.size else 0.0
    active_M_late = float(M_hist[late_slice][:, active_mask].mean())
    control_M_late = M_hist[late_slice][:, ~active_mask]
    control_max_M_late = float(control_M_late.mean(axis=0).max()) if control_M_late.size else 0.0

    dock_df = pd.DataFrame(dock_rows)
    pool_strong_fraction = float(np.mean(strong))
    pool_productive_weight_mean = float(np.mean(q_g))
    if len(dock_df):
        adf = dock_df[dock_df["active_synapse"] == 1]
        active_docks = int(len(adf)); n_productive = int(dock_df["productive"].sum())
        active_productive_docks = int(adf["productive"].sum()) if active_docks else 0
        mean_material_distance_docked = float(np.sqrt(dock_df["d2"].astype(float)).mean())
    else:
        adf = pd.DataFrame(); active_docks = 0; n_productive = 0
        active_productive_docks = 0; mean_material_distance_docked = np.nan

    if active_docks > 0:
        active_strong_dock_fraction = float(adf["strong"].mean())
        active_productive_weight_mean = float(adf["q_ig"].mean())
    else:
        active_strong_dock_fraction = np.nan
        active_productive_weight_mean = np.nan

    metrics = {
        "seed": int(params.get("seed", seed)), "sim_seed": int(seed),
        "variant": variant_name, "label": variant["label"],
        "active_S_late": active_S_late, "control_max_S_late": control_max_S_late,
        "continuous_specificity": active_S_late - control_max_S_late,
        "active_M_late": active_M_late, "control_max_M_late": control_max_M_late,
        "material_memory_specificity": active_M_late - control_max_M_late,
        "active_consolidation": float(active_S_late >= float(p["S_threshold"])),
        "active_material_memory": float(active_M_late >= float(p["M_threshold"])),
        "crosstalk_rate": float(control_max_S_late >= float(p["S_threshold"])),
        "material_crosstalk_rate": float(control_max_M_late >= float(p["M_threshold"])),
        "n_docks": int(len(dock_df)), "n_productive_docks": n_productive,
        "active_docks": active_docks, "active_productive_docks": active_productive_docks,
        "pool_strong_fraction": pool_strong_fraction,
        "active_strong_dock_fraction": active_strong_dock_fraction,
        "active_strong_enrichment": active_strong_dock_fraction / pool_strong_fraction
            if active_docks > 0 and pool_strong_fraction > 0 else np.nan,
        "pool_productive_weight_mean": pool_productive_weight_mean,
        "active_productive_weight_mean": active_productive_weight_mean,
        "active_productive_enrichment": active_productive_weight_mean / pool_productive_weight_mean
            if active_docks > 0 and pool_productive_weight_mean > 0 else np.nan,
        "mean_material_distance_docked": mean_material_distance_docked,
        # drift bookkeeping
        "tau_m": tau_m, "sigma_rel": sigma_rel, "sigma_m": sigma_m,
        "sigma_inf": sigma_inf, "ou_a": ou_a, "ou_b": ou_b, "dt_over_tau_m": (dt / tau_m) if drift_u_on else 0.0,
        "k_c": k_c, "n_copies": int(dr["n_copies"]),
    }
    for key in ["Nsyn", "Ngran", "L", "granule_density", "v", "D", "epsilon", "h0",
                "beta_T", "beta_M", "ell_h", "ell_q", "alphaS", "q_min", "tau_T",
                "tau_S", "delta_translation", "k_M", "alphaM", "K_M", "n_hill",
                "strong_sigma", "weak_shift", "weak_sigma", "strong_fraction"]:
        metrics[key] = p.get(key, np.nan)

    return {"metrics": metrics, "dock_df": dock_df, "obs_df": pd.DataFrame(obs_rows),
            "S_history": S_hist if record_trace else None,
            "M_history": M_hist if record_trace else None,
            "T_active_history": T_active_hist if record_trace else None,
            "xsyn": xsyn, "params": p}


def encounter_interval_stats(params: Dict[str, Any], seed: int,
                             n_probe: int = 1) -> float:
    """Median inter-encounter interval (s) for a granule, from geometry+transport."""
    p = dict(params)
    rng = np.random.default_rng(int(seed) + 555)
    L = float(p["L"]); Nsyn = int(p["Nsyn"]); dt = float(p["dt"])
    xsyn = np.linspace(L / (Nsyn + 1), L - L / (Nsyn + 1), Nsyn)
    n_steps = int(round(float(p["T_total"]) / dt))
    Ng = min(40, int(p["Ngran"]))
    gpos = rng.uniform(0.0, L, size=Ng); gdir = rng.choice([-1.0, 1.0], size=Ng)
    last = np.full(Ng, -1.0); gaps: List[float] = []
    for step in range(n_steps):
        t = step * dt
        gpos += gdir * float(p["v"]) * dt + np.sqrt(2.0 * float(p["D"]) * dt) * rng.standard_normal(Ng)
        gdir[rng.random(Ng) < float(p["p_switch"])] *= -1.0
        below = gpos < 0.0; above = gpos > L
        gpos[below] = -gpos[below]; gpos[above] = 2.0 * L - gpos[above]
        gdir[below | above] *= -1.0
        hit = (np.abs(gpos[:, None] - xsyn[None, :]) <= float(p["epsilon"])).any(axis=1)
        for g in np.flatnonzero(hit):
            if last[g] >= 0 and (t - last[g]) > dt * 1.5:
                gaps.append(t - last[g])
            last[g] = t
    return float(np.median(gaps)) if gaps else float("nan")


## Cell 3b — Verify the module and the exact-OU patch

In [ ]:
mp = PROJECT / 'msc_material_dynamics.py'
assert mp.exists(), f"module not found at {PROJECT} — edit the %%writefile path in Cell 3"
src = mp.read_text()
assert 'ou_a = math.exp(-dt / tau_m)' in src, 'exact-OU patch missing'
assert 'u_true = u0 + (u_true - u0) * ou_a' in src, 'exact-OU step missing'
print('OK: module', mp.stat().st_size, 'bytes, exact OU confirmed')

## Cell 4 — Imports

In [ ]:
import importlib
import msc_model_feedback as MM; importlib.reload(MM)
import msc_material_dynamics as MD; importlib.reload(MD)

from msc_model_feedback import (load_config, sample_params, simulate_one,
                                summarize_metrics, paired_effects, MODEL_VARIANTS)
from msc_material_dynamics import (simulate_one_drift, encounter_interval_stats,
                                   DRIFT_DEFAULTS)

CONFIG = load_config(PROJECT / 'config_msc_feedback.yaml')
print('variants:', list(MODEL_VARIANTS))

## Cell 5 — Seeding helper (matches `run_msc_feedback_colab.py`)

In [ ]:
def vseed(seed, variant):
    off = sum((j + 1) * ord(ch) for j, ch in enumerate(variant)) % 10000
    return seed * 100 + off

BASES = {s: sample_params(1000 + s, CONFIG) for s in range(N_SEEDS)}
pd.DataFrame([{k: BASES[s][k] for k in
               ('L','Nsyn','Ngran','v','D','epsilon','h0','beta_T','ell_h',
                'ell_q','strong_sigma','weak_shift','T_total','dt')}
              for s in range(N_SEEDS)]).round(4)

## Cell 6 — REGRESSION TEST (hard gate)

In [ ]:
keys = ['continuous_specificity', 'material_memory_specificity',
        'active_productive_enrichment', 'active_strong_enrichment',
        'n_docks', 'n_productive_docks', 'active_docks']
rows = []
for s in range(min(4, N_SEEDS)):
    sd = vseed(1000 + s, VARIANT)
    a = simulate_one(BASES[s], seed=sd, variant_name=VARIANT)['metrics']
    b = simulate_one_drift(BASES[s], seed=sd, variant_name=VARIANT)['metrics']
    for k in keys:
        va, vb = a[k], b[k]
        ok = (np.isnan(va) and np.isnan(vb)) or np.isclose(va, vb, rtol=0, atol=0)
        rows.append(dict(seed=1000+s, metric=k, ref=va, drift_off=vb, identical=ok))
chk = pd.DataFrame(rows)
assert chk['identical'].all(), chk[~chk['identical']]
print('REGRESSION PASSED — drift-off is bit-identical')
chk.head(14)

## Cell 6b — OU INTEGRATOR TEST (hard gate)

In [ ]:
def ou_stationary_sd(tau_m, dt=2.0, n=200000, seed=1):
    r = np.random.default_rng(seed)
    a = math.exp(-dt / tau_m); b = math.sqrt(max(0.0, 1 - a * a))
    x = 0.0; out = np.empty(n)
    for i in range(n):
        x = x * a + b * r.standard_normal(); out[i] = x
    return float(out[n // 2:].std())

ou = pd.DataFrame([dict(tau_m=tm, dt_over_tau_m=2.0 / tm, sd=ou_stationary_sd(tm))
                   for tm in [0.02, 0.2, 2.0, 20.0, 200.0]])
ou['ok'] = (ou['sd'] - 1.0).abs() < 0.05
assert ou['ok'].all(), ou
print('OU INTEGRATOR PASSED — sigma_inf recovered at every timescale')
ou.round(4)

## Cell 7 — Calibrate tau_enc per seed

In [ ]:
TAU_ENC = {s: encounter_interval_stats(BASES[s], 1000 + s) for s in range(N_SEEDS)}
json.dump({str(k): v for k, v in TAU_ENC.items()},
          open(DATA / 'tau_enc_per_seed.json', 'w'), indent=2)
te = pd.Series(TAU_ENC, name='tau_enc_s')
print(te.describe().round(2))
print('\ndt / tau_enc per seed:',
      {s: round(float(BASES[s]['dt']) / TAU_ENC[s], 3) for s in range(N_SEEDS)})
te

## Cell 8 — Reference controls at zero drift

In [ ]:
# Reuse a complete block on rerun. Delete the CSV to recompute it.
_cache_path=DATA/'msc_drift_reference_controls_v7.csv'
if _cache_path.exists() and _cache_path.stat().st_size>0:
    ref=pd.read_csv(_cache_path)
    print('Reusing completed block:',_cache_path.name,ref.shape)
else:
    REF_VARIANTS = ['full_msc_feedback', 'msc_capture_only', 'projected1d_feedback',
                    'shuffled_material_feedback', 'random_material_feedback',
                    'tag_only_feedback']

    def run_ref(s, v):
        return simulate_one_drift(BASES[s], seed=vseed(1000+s, v), variant_name=v)['metrics']

    ref = pd.DataFrame(Parallel(n_jobs=N_JOBS, verbose=5)(
        delayed(run_ref)(s, v) for s in range(N_SEEDS) for v in REF_VARIANTS))
    ref.to_csv(DATA / 'msc_drift_reference_controls_v7.csv', index=False)

    ref.groupby('label')[['continuous_specificity','material_memory_specificity',
                          'active_strong_enrichment','active_productive_enrichment',
                          'crosstalk_rate','material_crosstalk_rate','n_docks']].mean().round(4)


## Cell 9 — Panels A–C: granule material drift (PRIMARY)

Grid refined around rho = 1 so the dip position can be quoted rather than bracketed to a decade. Each run records `dt_over_tau_m`; points above 1 are flagged `dt_resolved = False` because the simulation cannot represent drift faster than one timestep.

In [ ]:
# Reuse a complete block on rerun. Delete the CSV to recompute it.
_cache_path=DATA/'msc_granule_drift_sweep_v7.csv'
if _cache_path.exists() and _cache_path.stat().st_size>0:
    dr=pd.read_csv(_cache_path)
    print('Reusing completed block:',_cache_path.name,dr.shape)
else:
    RHO_GRID = ([0.03, 0.1, 0.3, 1, 3, 10, 30] if QUICK
                else [0.003, 0.01, 0.03, 0.1, 0.3, 0.5, 1, 2, 3, 5, 10, 30, 100, 300])
    SIGREL_GRID = [0.25, 0.50, 1.00]

    def run_drift(s, rho, sr):
        tm = rho * TAU_ENC[s]
        m = simulate_one_drift(BASES[s], seed=vseed(1000+s, VARIANT), variant_name=VARIANT,
                               drift=dict(tau_m=tm, sigma_rel=sr))['metrics']
        m['rho'] = rho; m['tau_enc'] = TAU_ENC[s]
        m['tau_m_over_Ttotal'] = tm / float(BASES[s]['T_total'])
        m['equilibrated'] = m['tau_m_over_Ttotal'] <= EQUIL_MAX
        m['dt_resolved'] = m['dt_over_tau_m'] <= 1.0
        return m

    def run_nodrift(s):
        m = simulate_one_drift(BASES[s], seed=vseed(1000+s, VARIANT),
                               variant_name=VARIANT)['metrics']
        m['rho'] = np.inf; m['tau_enc'] = TAU_ENC[s]
        m['tau_m_over_Ttotal'] = np.inf; m['equilibrated'] = False; m['dt_resolved'] = True
        return m

    t0 = time.time()
    dr = pd.DataFrame(Parallel(n_jobs=N_JOBS, verbose=5)(
        [delayed(run_drift)(s, r, sr)
         for s in range(N_SEEDS) for r in RHO_GRID for sr in SIGREL_GRID]
        + [delayed(run_nodrift)(s) for s in range(N_SEEDS)]))
    dr.to_csv(DATA / 'msc_granule_drift_sweep_v7.csv', index=False)
    print('elapsed', round(time.time()-t0, 1), 's |', dr.shape)
    print('NaN rows:', int(dr['continuous_specificity'].isna().sum()))
    print('max dt/tau_m visited:', round(float(dr.query('rho != inf')['dt_over_tau_m'].max()), 2))
    print(dr.query('rho != inf').groupby('rho')[['equilibrated','dt_resolved']].mean().round(2))


## Cell 9b — dt CONTROL for the fast branch (decisive test)

The fast-drift recovery in v5 may be a discretization ceiling rather than motional narrowing. Here the same fast-branch conditions are re-run at half the timestep, at the largest amplitude where the effect is clearest.

Interpretation: if the paired difference between full dt and half dt is within noise, the fast branch is resolved and motional narrowing can be claimed. If half dt gives systematically **lower** specificity, the recovery is an artifact and only rho >= 0.1 should be reported.

In [ ]:
# Reuse a complete block on rerun. Delete the CSV to recompute it.
_cache_path=DATA/'msc_dt_control_v7.csv'
if _cache_path.exists() and _cache_path.stat().st_size>0:
    dtc=pd.read_csv(_cache_path)
    print('Reusing completed block:',_cache_path.name,dtc.shape)
else:
    FAST_RHO = [0.03, 0.1, 0.3, 1.0]
    DT_FACTORS = [1.0, 0.5]
    SR_TEST = 1.00

    def run_dt(s, rho, f):
        b = dict(BASES[s]); b['dt'] = float(BASES[s]['dt']) * f
        m = simulate_one_drift(b, seed=vseed(1000+s, VARIANT), variant_name=VARIANT,
                               drift=dict(tau_m=rho * TAU_ENC[s], sigma_rel=SR_TEST))['metrics']
        m['rho'] = rho; m['dt_factor'] = f
        return m

    def run_dt_base(s, f):
        b = dict(BASES[s]); b['dt'] = float(BASES[s]['dt']) * f
        m = simulate_one_drift(b, seed=vseed(1000+s, VARIANT), variant_name=VARIANT)['metrics']
        m['rho'] = np.inf; m['dt_factor'] = f
        return m

    dtc = pd.DataFrame(Parallel(n_jobs=N_JOBS, verbose=5)(
        [delayed(run_dt)(s, r, f) for s in range(N_SEEDS)
         for r in FAST_RHO for f in DT_FACTORS]
        + [delayed(run_dt_base)(s, f) for s in range(N_SEEDS) for f in DT_FACTORS]))
    dtc.to_csv(DATA / 'msc_dt_control_v7.csv', index=False)

    # normalize within (seed, dt_factor) to that timestep's own no-drift run
    base = (dtc[dtc.rho == np.inf].set_index(['seed','dt_factor'])['continuous_specificity'])
    dtn = dtc[dtc.rho != np.inf].copy()
    dtn['spec_rel'] = dtn['continuous_specificity'].values / \
                    dtn.set_index(['seed','dt_factor']).index.map(base).values

    tab = dtn.pivot_table(index=['seed','rho'], columns='dt_factor', values='spec_rel')
    tab['diff_half_minus_full'] = tab[0.5] - tab[1.0]
    by_rho = tab.groupby('rho')['diff_half_minus_full'].agg(['mean','sem','count'])
    print('spec_rel(dt/2) - spec_rel(dt), paired within seed:')
    print(by_rho.round(4))

    verdict = {}
    for rho, r in by_rho.iterrows():
        sig = abs(r['mean']) > 2 * r['sem'] if r['sem'] > 0 else False
        verdict[float(rho)] = 'DT-LIMITED' if (sig and r['mean'] < 0) else 'resolved'
        print(f"rho={rho:g}: {verdict[float(rho)]}  (mean {r['mean']:+.4f} +/- {r['sem']:.4f})")
    DT_OK = all(v == 'resolved' for v in verdict.values())
    print('\nfast branch fully resolved:', DT_OK)
    if not DT_OK:
        print('-> restrict the motional-narrowing claim to dt-resolved rho only')


## Cell 10 — Paired within-seed normalization

In [ ]:
base_by_seed = (dr.query('rho == inf')
                  .set_index('seed')[['continuous_specificity',
                                      'active_strong_enrichment',
                                      'material_memory_specificity']])
sw = dr.query('rho != inf').copy()
for col, new in [('continuous_specificity', 'spec_rel'),
                 ('active_strong_enrichment', 'strongenr_rel'),
                 ('material_memory_specificity', 'Mspec_rel')]:
    ref_v = sw['seed'].map(base_by_seed[col])
    sw[new] = np.where(np.abs(ref_v) > 1e-9, sw[col] / ref_v, np.nan)

def boot_paired(x, n=5000, seed=0):
    x = np.asarray(pd.Series(x).replace([np.inf,-np.inf], np.nan).dropna())
    if len(x) < 2: return (np.nan, np.nan)
    r = np.random.default_rng(seed)
    return tuple(np.percentile(r.choice(x, (n, len(x)), replace=True).mean(1), [2.5, 97.5]))

rows = []
for (sr, rho), g in sw.groupby(['sigma_rel', 'rho']):
    lo, hi = boot_paired(g['spec_rel'])
    rows.append(dict(sigma_rel=sr, rho=rho,
                     spec_rel_mean=g['spec_rel'].mean(), lo=lo, hi=hi,
                     strongenr_rel=g['strongenr_rel'].mean(),
                     Mspec_rel=g['Mspec_rel'].mean(),
                     dt_over_tau_m=g['dt_over_tau_m'].mean(),
                     dt_resolved=bool(g['dt_resolved'].all()),
                     equilibrated=bool(g['equilibrated'].all()), n=len(g)))
paired = pd.DataFrame(rows).sort_values(['sigma_rel','rho'])
paired.to_csv(DATA / 'msc_drift_paired_v7.csv', index=False)
paired.round(4)

## Cell 11 — Dip localization and floor test

v5 showed no crossing of the random-material floor, so rho* is undefined and the reportable quantity is the **position and depth of the minimum**. The dip is located by parabolic interpolation through the three points bracketing the grid minimum in log10(rho). The floor test is retained so its absence is documented rather than assumed.

In [ ]:
floor_abs = ref.query("variant=='random_material_feedback'")['continuous_specificity']
full_abs  = ref.query("variant=='full_msc_feedback'")['continuous_specificity']
floor_rel = float((floor_abs.values / full_abs.values).mean())
print(f'random-material floor on relative scale = {floor_rel:.4f}\n')

def dip(g):
    g = g.sort_values('rho'); x = np.log10(g.rho.values); y = g.spec_rel_mean.values
    i = int(y.argmin())
    if i in (0, len(y) - 1):
        return dict(rho_dip=float(10 ** x[i]), depth=float(1 - y[i]), edge=True)
    x0, x1, x2 = x[i-1:i+2]; y0, y1, y2 = y[i-1:i+2]
    den = (y0 - 2*y1 + y2)
    xv = x1 + 0.5 * (y0 - y2) / den * (x1 - x0) if abs(den) > 1e-12 else x1
    return dict(rho_dip=float(10 ** xv), depth=float(1 - y1), edge=False)

dips = {}
for sr, g in paired.groupby('sigma_rel'):
    dips[sr] = dip(g)
    ge = g[g.lo > floor_rel]
    reaches_floor = len(ge) < len(g)
    dinfo = dips[sr]
    print(f"sigma_rel={sr}: dip at rho = {dinfo['rho_dip']:.2f}, depth = {dinfo['depth']*100:.1f}% "
          f"{'(EDGE — grid too narrow)' if dinfo['edge'] else ''}")
    print(f"          floor reached anywhere: {reaches_floor}  -> "
          f"rho* {'defined' if reaches_floor else 'UNDEFINED (report resonance, not threshold)'}")
    sig = g[(g.hi < 1.0)]
    print(f"          rho with CI excluding 1.0: {list(sig.rho.round(3)) or 'none'}\n")

paired.round(4)

## Cell 12 — Panel D: transcript copy number

In [ ]:
# Reuse a complete block on rerun. Delete the CSV to recompute it.
_cache_path=DATA/'msc_copynumber_v7.csv'
if _cache_path.exists() and _cache_path.stat().st_size>0:
    cp=pd.read_csv(_cache_path)
    print('Reusing completed block:',_cache_path.name,cp.shape)
else:
    COPIES = [1, 3, 10]

    def run_copy(s, n):
        return simulate_one_drift(BASES[s], seed=vseed(1000+s, VARIANT),
                                  variant_name=VARIANT, drift=dict(n_copies=n))['metrics']

    cp = pd.DataFrame(Parallel(n_jobs=N_JOBS, verbose=5)(
        delayed(run_copy)(s, n) for s in range(N_SEEDS) for n in COPIES))
    cp.to_csv(DATA / 'msc_copynumber_v7.csv', index=False)

    piv = cp.pivot_table(index='seed', columns='n_copies', values='continuous_specificity')
    for n in COPIES[1:]:
        diff_n = (piv[n] - piv[1]).dropna()
        if len(diff_n) >= 5 and diff_n.abs().sum() > 0:
            print(f'N={n} vs N=1: mean diff {diff_n.mean():+.4f}, '
                  f'Wilcoxon p={wilcoxon(piv[n], piv[1]).pvalue:.4g}')

    sc = pd.DataFrame({'ell_h': [BASES[s]['ell_h'] for s in range(N_SEEDS)],
                       'strong_sigma': [BASES[s]['strong_sigma'] for s in range(N_SEEDS)]})
    sc['ratio'] = sc['ell_h'] / sc['strong_sigma']
    print('\nell_h / strong_sigma across seeds:',
          sc['ratio'].describe()[['mean','min','max']].round(2).to_dict())

    cprel = piv.div(piv[1], axis=0)
    cp.groupby('n_copies')[['continuous_specificity','active_strong_enrichment',
                            'active_productive_enrichment','n_docks']].agg(['mean','sem']).round(4)


## Cell 13 — Panel E: PSD target drift (robustness)

In [ ]:
# Reuse a complete block on rerun. Delete the CSV to recompute it.
_cache_path=DATA/'msc_psd_target_drift_v7.csv'
if _cache_path.exists() and _cache_path.stat().st_size>0:
    ps=pd.read_csv(_cache_path)
    print('Reusing completed block:',_cache_path.name,ps.shape)
else:
    KC_GRID = [0.0, 1e-5, 1e-4, 1e-3, 1e-2]
    N_SEC   = max(4, N_SEEDS // 2)

    def run_psd(s, kc):
        return simulate_one_drift(BASES[s], seed=vseed(1000+s, VARIANT),
                                  variant_name=VARIANT, drift=dict(k_c=kc))['metrics']

    ps = pd.DataFrame(Parallel(n_jobs=N_JOBS, verbose=5)(
        delayed(run_psd)(s, kc) for s in range(N_SEC) for kc in KC_GRID))
    ps.to_csv(DATA / 'msc_psd_target_drift_v7.csv', index=False)

    pk = ps.pivot_table(index='seed', columns='k_c', values='continuous_specificity')
    rel = pk.div(pk[0.0], axis=0)
    print('paired, relative to k_c = 0:'); print(rel.mean().round(4))
    ps.groupby('k_c')[['continuous_specificity','material_memory_specificity',
                       'active_strong_enrichment']].agg(['mean','sem']).round(4)


## Cell 14 — Figure S-x (panels A–F)

Panel B now overlays docking-stage selectivity (strong-granule enrichment) against the material-memory readout, making the dissociation explicit. Panel C is replaced by the dt control, which is the load-bearing evidence for the fast branch; the floor comparison moves into panel A.

In [ ]:
# self-contained: rebuild the dt-control frame from disk so this cell
# never depends on variables left over from earlier cells
dtc = pd.read_csv(DATA / 'msc_dt_control_v7.csv')
_b  = dtc[dtc.rho == np.inf].set_index(['seed','dt_factor'])['continuous_specificity']
dtn = dtc[dtc.rho != np.inf].copy()
dtn['spec_rel'] = dtn['continuous_specificity'].values / \
                  dtn.set_index(['seed','dt_factor']).index.map(_b).values

fig, ax = plt.subplots(2, 3, figsize=(16, 9))
COL = {0.25:'#2c7fb8', 0.50:'#41ab5d', 1.00:'#d95f0e'}

a = ax[0,0]
for sr, g in paired.groupby('sigma_rel'):
    g = g.sort_values('rho')
    a.plot(g.rho, g.spec_rel_mean, marker='o', color=COL[sr], label=f'$\\sigma_{{rel}}$={sr}')
    a.fill_between(g.rho, g.lo, g.hi, color=COL[sr], alpha=0.18)
    nr = g[~g.dt_resolved]
    if len(nr): a.scatter(nr.rho, nr.spec_rel_mean, marker='x', s=70, c='k', zorder=5)
    a.axvline(dips[sr]['rho_dip'], color=COL[sr], ls='-.', lw=0.8, alpha=0.55)
a.axhline(1.0, ls='--', c='k', lw=1, label='static full MSC')
a.axhline(floor_rel, ls=':', c='r', lw=1, label='random-material floor')
a.set_xscale('log'); a.set_xlabel(r'$\rho=\tau_m/\tau_{enc}$')
a.set_ylabel('relative consolidation specificity')
a.set_title('(A) Resonant degradation at $\\rho\\approx1$')
a.legend(fontsize=7); a.text(0.02,0.02,'x = dt-unresolved',transform=a.transAxes,fontsize=6)

a = ax[0,1]
for sr, g in paired.groupby('sigma_rel'):
    g = g.sort_values('rho')
    a.plot(g.rho, g.strongenr_rel, marker='s', ls='-', color=COL[sr],
           label=f'docking, $\\sigma_{{rel}}$={sr}')
    a.plot(g.rho, g.Mspec_rel, marker='^', ls=':', color=COL[sr], alpha=0.75,
           label=f'$M$ tag, $\\sigma_{{rel}}$={sr}')
a.axhline(1.0, ls='--', c='k', lw=1)
a.set_xscale('log'); a.set_xlabel(r'$\rho$'); a.set_ylabel('relative to static MSC')
a.set_title('(B) Docking filter vs material memory'); a.legend(fontsize=6, ncol=2)

a = ax[0,2]
for f, mk in [(1.0,'o'), (0.5,'s')]:
    gg = dtn[dtn.dt_factor == f].groupby('rho')['spec_rel'].agg(['mean','sem'])
    a.errorbar(gg.index, gg['mean'], yerr=gg['sem'], marker=mk,
               label=f'dt x {f:g}', capsize=3)
a.axhline(1.0, ls='--', c='k', lw=1)
a.set_xscale('log'); a.set_xlabel(r'$\rho$')
a.set_ylabel('relative specificity')
a.set_title('(C) Timestep control, $\\sigma_{rel}$=1.0'); a.legend(fontsize=7)

a = ax[1,0]
bar_d = cp.groupby('n_copies')['active_strong_enrichment'].agg(['mean','sem'])
a.bar([str(i) for i in bar_d.index], bar_d['mean'], yerr=bar_d['sem'], color='#41ab5d', capsize=4)
a.axhline(1.0, ls=':', c='r', lw=1)
a.set_xlabel('transcript copies per granule $N$')
a.set_ylabel('strong-granule enrichment'); a.set_title('(D) Copy-number variance')

a = ax[1,1]
a.violinplot([cprel[n].dropna().values for n in COPIES], showmeans=True)
a.axhline(1.0, ls=':', c='r', lw=1)
a.set_xticks(range(1, len(COPIES)+1)); a.set_xticklabels([str(n) for n in COPIES])
a.set_xlabel('copies $N$'); a.set_ylabel('specificity relative to $N=1$')
a.set_title("(E) Paired, within-seed")

a = ax[1,2]
m = rel.mean(); e = rel.sem()
a.errorbar(range(len(m)), m.values, yerr=e.values, marker='o', color='#756bb1')
a.set_xticks(range(len(m))); a.set_xticklabels([f'{k:g}' for k in m.index], rotation=45)
a.axhline(1.0, ls='--', c='k', lw=1); a.axvspan(-0.4, 2.4, color='green', alpha=0.08)
a.set_xlabel('$k_c$  (green = physiological regime)')
a.set_ylabel('specificity relative to $k_c=0$')
a.set_title('(F) PSD target drift — robustness')

plt.tight_layout()
plt.savefig(FIG / 'FigS_material_dynamics_v7.png', dpi=300, bbox_inches='tight')
plt.show()

a.set_title('(A) Drift amplitude, not timescale, sets specificity loss')
a.set_title('(B) Docking filter preserved, material memory degraded')
a.set_title('(C) Fast branch is timestep-limited')
a.set_title('(D) Copy number does not alter selectivity')
a.set_title('(E) Same, paired within seed')
a.set_title('(F) Robust to activity-dependent PSD drift')

## Cell 15 — Projected-1D normalization check

In [ ]:
cmp = ref.query("variant in ['full_msc_feedback','projected1d_feedback']")
p1 = cmp.pivot_table(index='seed', columns='variant',
                     values=['continuous_specificity','mean_material_distance_docked','n_docks'])
print(p1.round(4))
frac = float((p1[('continuous_specificity','projected1d_feedback')]
              > p1[('continuous_specificity','full_msc_feedback')]).mean())
print(f'\nfraction of seeds where projected-1D EXCEEDS full MSC: {frac:.2f}')
print('mean docked d^2  full:',
      round(p1[('mean_material_distance_docked','full_msc_feedback')].mean(), 4),
      '| projected1d:',
      round(p1[('mean_material_distance_docked','projected1d_feedback')].mean(), 4))
if frac > 0.5:
    print('\nCONFOUND CONFIRMED — 1-axis projection lowers mean d^2 and raises docking.')
    print('Report as a normalization artifact of the control, not as selectivity.')

## Cell 16 — Manuscript-ready numbers

In [ ]:
summary = {
    'version': 'v7_resonance', 'n_seeds': N_SEEDS, 'quick_mode': QUICK,
    'tau_enc_median_s': float(np.median(list(TAU_ENC.values()))),
    'tau_enc_range_s': [float(min(TAU_ENC.values())), float(max(TAU_ENC.values()))],
    'floor_relative': round(floor_rel, 4),
    'dip': {str(k): {kk: round(vv, 4) if isinstance(vv, float) else vv
                     for kk, vv in v.items()} for k, v in dips.items()},
    'rho_star': 'undefined — floor never reached',
    'dt_control_verdict': verdict,
    'fast_branch_resolved': bool(DT_OK),
    'spec_rel_by_rho': {str(sr): dict(zip(g.rho.astype(str), g.spec_rel_mean.round(4)))
                        for sr, g in paired.groupby('sigma_rel')},
    'strongenr_rel_by_rho': {str(sr): dict(zip(g.rho.astype(str), g.strongenr_rel.round(4)))
                             for sr, g in paired.groupby('sigma_rel')},
    'Mspec_rel_by_rho': {str(sr): dict(zip(g.rho.astype(str), g.Mspec_rel.round(4)))
                         for sr, g in paired.groupby('sigma_rel')},
    'copy_number_strong_enrichment':
        cp.groupby('n_copies')['active_strong_enrichment'].mean().round(4).to_dict(),
    'copy_number_specificity_relative': cprel.mean().round(4).to_dict(),
    'psd_drift_relative': rel.mean().round(4).to_dict(),
    'reference_controls':
        ref.groupby('variant')['continuous_specificity'].mean().round(4).to_dict(),
    'projected1d_exceeds_full_fraction': frac,
}
json.dump(summary, open(DATA / 'summary_material_dynamics_v7.json', 'w'), indent=2)
print(json.dumps(summary, indent=2))

## Cell 17 — Figure legend and Limitations draft

Insert the dip positions from Cell 11 and the dt verdict from Cell 9b before using this text.

**Figure S-x. Time-varying material identity in the MSC model.** **(A)** Consolidation specificity, normalized within seed to the corresponding static simulation, against the dimensionless ratio rho = tau_m/tau_enc under Ornstein–Uhlenbeck drift of the granule material vector, at three fixed stationary drift amplitudes sigma_rel. Shaded bands, paired bootstrap 95% CI; dash-dotted vertical lines, interpolated position of the minimum; crosses mark points at which the drift timescale falls below the integration timestep. Dashed line, static full MSC; dotted line, random-material control. **(B)** Docking-stage material selectivity (strong-granule enrichment, solid) and the material-memory readout (dotted), both normalized within seed, showing that the compatibility filter is preserved where the downstream tag is degraded. **(C)** Timestep control at sigma_rel = 1.0: the fast branch re-run at half the integration step. **(D)** Strong-granule enrichment for granules carrying 1, 3 or 10 transcript copies, where copy number rescales material variance as sigma/sqrt(N) with the weak-population offset held fixed. **(E)** Paired within-seed specificity relative to N = 1. **(F)** Consolidation specificity relative to k_c = 0 under activity-dependent drift of the synaptic target vector; shading marks the physiological k_c regime implied by measured postsynaptic scaffold turnover. Points show means, error bars s.e.m. across seeds.

---

### Suggested Results wording (resonance framing)

> Relaxing the quasi-static assumption on granule material identity produced a non-monotone response. Consolidation specificity was maximally reduced when the material relaxation time was comparable to the granule–synapse encounter interval, and recovered toward the static model in both the fast and slow limits: at the largest drift amplitude tested the model retained [X]% of static specificity at rho << 1 and [Y]% at rho >> 1, against a minimum of [Z]% near rho ~ 1. Specificity did not approach the random-material control at any drift timescale, so the model predicts degradation rather than abolition of material selectivity. Notably, docking-stage strong-granule enrichment remained within [W]% of its static value throughout, while the material-memory tag declined in parallel with consolidation specificity, indicating that drift on the encounter timescale corrupts the temporal coherence of repeated productive capture rather than the compatibility filter itself.

### Suggested Limitations paragraph

> Granule and synaptic material vectors are treated as quasi-static on the capture timescale in the main simulations. Supplementary analysis indicates that this assumption is not load-bearing in either asymptotic limit, but that MSC selectivity is measurably reduced when material identity fluctuates on the encounter timescale itself. The magnitude of this reduction depends on the amplitude of material fluctuation, which is not independently constrained by available data. The mapping between the model drift constant and physical time likewise remains a calibration assumption rather than a measurement; direct estimates of RNP material relaxation times, for example from FRAP recovery kinetics or single-particle tracking of tagged granules, would be required to place real synapses on this axis. The fast-drift regime is additionally bounded by the integration timestep, and conclusions there rest on the timestep control reported in panel C.

## Cell 18 — Canonical exports, manifest and single reproducibility bundle

In [ ]:
# 18. Canonical exports, manifest and single reproducibility bundle
required_files=[DATA/'tau_enc_per_seed.json',DATA/'msc_drift_reference_controls_v7.csv',
                DATA/'msc_granule_drift_sweep_v7.csv',DATA/'msc_dt_control_v7.csv',
                DATA/'msc_copynumber_v7.csv',DATA/'msc_psd_target_drift_v7.csv',
                DATA/'summary_material_dynamics_v7.json']
missing=[str(p) for p in required_files if not p.exists() or p.stat().st_size<=0]
if missing: raise RuntimeError('Required output files missing:\n'+'\n'.join(missing))

drift_table=pd.read_csv(DATA/'msc_granule_drift_sweep_v7.csv')
dt_table=pd.read_csv(DATA/'msc_dt_control_v7.csv')
copy_table=pd.read_csv(DATA/'msc_copynumber_v7.csv')
psd_table=pd.read_csv(DATA/'msc_psd_target_drift_v7.csv')
assert set(drift_table.sigma_rel)=={0.25,0.5,1.0}
assert set(copy_table.n_copies)=={1,3,10}
assert set(psd_table.k_c)=={0.0,1e-5,1e-4,1e-3,1e-2}
assert len(set(dt_table.seed))==N_SEEDS

for path in required_files: shutil.copy2(path,R_INPUT/path.name)
aliases={'panel_A_granule_material_drift.csv':'msc_granule_drift_sweep_v7.csv',
         'panel_B_docking_memory_dissociation.csv':'msc_granule_drift_sweep_v7.csv',
         'panel_C_timestep_control.csv':'msc_dt_control_v7.csv',
         'panel_D_copy_number_enrichment.csv':'msc_copynumber_v7.csv',
         'panel_E_copy_number_specificity.csv':'msc_copynumber_v7.csv',
         'panel_F_psd_target_drift.csv':'msc_psd_target_drift_v7.csv'}
for target,source in aliases.items(): shutil.copy2(DATA/source,R_INPUT/target)
for path in FIG.glob('*'):
    if path.is_file() and path.stat().st_size>0: shutil.copy2(path,R_INPUT/path.name)

manifest={'analysis':'MSC_time_varying_material_identity','version':'v7_exact_OU_reproducibility',
 'run_mode':'smoke' if QUICK else 'production','model_variant':VARIANT,'n_seeds':int(N_SEEDS),
 'design':{'rho_definition':'tau_m/tau_enc; tau_enc calibrated per seed','rho_grid':[float(x) for x in RHO_GRID],
           'sigma_rel_grid':[float(x) for x in SIGREL_GRID],'copy_number_grid':[int(x) for x in COPIES],
           'psd_target_drift_grid_s_inverse':[float(x) for x in KC_GRID],'exact_OU':True,
           'identity_reference':'granule birth material coordinate','productive_capacity':'fixed at birth'},
 'validation':{'regression_bit_identical_when_drift_off':bool(chk.identical.all()),
               'fast_branch_resolved':bool(DT_OK),'dt_control_verdict':verdict,
               'drift_rows':int(len(drift_table)),'dt_rows':int(len(dt_table)),
               'copy_rows':int(len(copy_table)),'psd_drift_rows':int(len(psd_table))},
 'model_sha256':sha256_file(PROJECT/'msc_model_feedback.py'),
 'config_sha256':sha256_file(PROJECT/'config_msc_feedback.yaml'),
 'extension_sha256':sha256_file(PROJECT/'msc_material_dynamics.py'),
 'generated_at':datetime.datetime.utcnow().isoformat()+'Z',
 'outputs':{str(p.relative_to(ROOT)):sha256_file(p) for p in sorted(ROOT.rglob('*')) if p.is_file()}}
(ROOT/'MANIFEST_material_dynamics.json').write_text(json.dumps(manifest,indent=2))

readme='''# MSC Supplementary Figure S1 reproducibility bundle

This bundle reproduces the v7 exact-Ornstein-Uhlenbeck material-dynamics analysis.
It tests bounded fluctuations around a granule birth material coordinate, transcript-copy
number averaging, and activity-dependent PSD-target drift. It does not model complete
loss or switching of material identity. Productive capacity is fixed at birth.

Production uses 100 matched parameter seeds. The ratio rho equals tau_m/tau_enc, where
tau_enc is calibrated per seed. The exact OU process has a separate random-number stream,
so drift-disabled simulations are bit-identical to the unmodified MSC model.

Full seed-level data and stable panel aliases for the later R renderer are in 10_r_input/.
All quantities are stochastic in-silico predictions, not wet-lab measurements.
'''
(ROOT/'README.md').write_text(readme)

zip_path=pathlib.Path('/content/MSC_SupplementaryFigureS1_material_dynamics_bundle.zip')
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(ROOT.rglob('*')):
        if path.is_file(): archive.write(path,path.relative_to(ROOT.parent))
assert zip_path.exists() and zip_path.stat().st_size>0
print('VALIDATION PASSED')
print('ZIP:',zip_path.name,f'{zip_path.stat().st_size/1e6:.1f} MB')
print('R inputs:',len(list(R_INPUT.iterdir())))
files.download(str(zip_path))
